# Playing interactive fiction inside a notebook

Poplog ships a **Z-machine** — the virtual machine Infocom shipped *Zork*
on in 1979 — written in Pop-11 (`LIB ZMACHINE`, about 1,500 lines across
seven files). It runs the same story files a 1980s home computer did, and
it passes the CZECH conformance suite for versions 3 and 5 without a
single failure.

This notebook plays a game in it. Not a recording of a game: the
interpreter is running in this notebook's kernel session, and each cell
below is one turn.

The story is *The Poplog Cave*, a miniature written for this project so
that something freely licensed ships with Poplog — every interactive
fiction actually worth playing belongs to somebody. Its source is
`examples/games/cave.inf`, about a hundred lines of library-free Inform 6.


## Starting the game

`zplay_start` loads the story, runs the machine up to its first prompt, and hands back everything it printed on the way.

In [1]:
uses zmachine_play;

;;; the story ships with Poplog, so $usepop finds it in a checkout
;;; and in an installed tree alike
zplay_start('$usepop/examples/games/cave.z3') =>

**
THE POPLOG CAVE
A miniature, written to exercise a Z-machine written in Pop-11.
Try: LOOK, NORTH, SOUTH, TAKE LAMP, LIGHT LAMP, INVENTORY, SCORE, QUIT.
Cave Mouth
A crack in the hillside opens north into the dark. Daylight lies south.
A brass lamp has been left here.
>


🌈 pop 82.32 ms │ session 101.45 ms over 2 runs │ all-time 27718.15 s


## How a turn works

`zplay_turn` looks like an ordinary procedure call, but the interpreter it
resumes is suspended *inside* the `sread` instruction, halfway through
executing the game's own code.

That is a Pop-11 **process** — a coroutine. The Z-machine runs inside one;
when the game asks for a command the process suspends, handing this cell
its output, and the next cell resumes it with the next command. Nothing
replays, nothing is re-entered, and the game's whole world — its object
tree, its stack, its program counter — simply waits between cells.

There is a pleasing circularity to this. Poplog's process machinery
depends on two assembly routines, `_ussave` and `_userasund`, which save
and restore the user stack. On arm64 they were unimplemented placeholder
stubs until a few days before this notebook was written; fixing them is
what made coroutines work on Apple Silicon at all — and therefore what
makes this cell work.


The cave mouth, described again — `look` is the cheapest way to see the parser working.

In [2]:
zplay_turn('look') =>

**
Cave Mouth
A crack in the hillside opens north into the dark. Daylight lies south.
A brass lamp has been left here.
>


🌈 pop 6.55 ms │ session 107.99 ms over 3 runs │ all-time 27718.16 s


## Light first

The chamber ahead is dark. Games have been teaching players to pick up the
lamp before going in since 1976.


In [3]:
zplay_turn('take lamp') =>

** Taken.
>


🌈 pop 6.50 ms │ session 114.49 ms over 4 runs │ all-time 27718.16 s


In [4]:
zplay_turn('light lamp') =>

** The lamp catches, and throws a small steady light.
>


🌈 pop 6.45 ms │ session 120.95 ms over 5 runs │ all-time 27718.17 s


In [5]:
zplay_turn('north') =>

**
Twisty Passage
The passage doubles back on itself. It runs north and south.
>


🌈 pop 6.48 ms │ session 127.43 ms over 6 runs │ all-time 27718.18 s


This is the dark chamber. With the lamp lit, it has a description.

In [6]:
zplay_turn('north') =>

**
Dark Chamber
Wet walls glisten in the lamplight. Ways lead north and south.
>


🌈 pop 5.63 ms │ session 133.06 ms over 7 runs │ all-time 27718.18 s


In [7]:
zplay_turn('north') =>

**
Treasure Room
A vaulted chamber, its ceiling lost above you. The way out is south.
A great gem rests on a pedestal.
>


🌈 pop 6.54 ms │ session 139.59 ms over 8 runs │ all-time 27718.19 s


In [8]:
zplay_turn('take gem') =>

** Taken. The cave seems to hold its breath.
>


🌈 pop 6.54 ms │ session 146.13 ms over 9 runs │ all-time 27718.20 s


In [9]:
zplay_turn('inventory') =>

** You are carrying:
  a brass lamp (lit)
  a great gem
>


🌈 pop 6.80 ms │ session 152.93 ms over 10 runs │ all-time 27718.20 s


In [10]:
zplay_turn('score') =>

** You have taken 9 turns and you are carrying the gem. You have won.
>


🌈 pop 6.64 ms │ session 159.57 ms over 11 runs │ all-time 27718.21 s


## What just happened

Every line of prose above was produced by Z-code — the compiled game —
being interpreted a byte at a time by Pop-11: `sread` tokenised the typed
command against the story's own dictionary, the game's routines walked its
object tree, and `print_obj` and friends unpacked text that is stored
three five-bit characters to a sixteen-bit word.

The same interpreter runs the real thing. Point `zplay_start` at any v3 or
v5 story file you own — *Zork I*, *Adventure*, anything from the IF
Archive — and it plays.

- The interpreter: `pop/lib/lib/zmachine*.p`
- How it was built, and why it is written the way it is:
  [`docs/projects/zmachine-design.md`](../../docs/projects/zmachine-design.md)
- Agents can play too: the `pop11_play` tool on Poplog's MCP server.
